In [1]:
# =========================================================
# FULL CNN + EVIDENTIAL + DEEP ENSEMBLE (CPU, JUPYTER)
# =========================================================

# ---------- IMPORTS ----------
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---------- PATHS ----------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\evidential_deep\GRU"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")

DEVICE = torch.device("cpu")

# ---------- SEED ----------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# ---------- LOAD DATA ----------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

# ---------- LABEL CLEANING (CORRECT FOR YOUR DATASET) ----------
# labels: "Non-sarcastic" / "Sarcastic"
train_df['labels'] = train_df['labels'].astype(str).str.strip().str.lower()
dev_df['labels']   = dev_df['labels'].astype(str).str.strip().str.lower()

label_map = {
    "non-sarcastic": 0,
    "sarcastic": 1
}

train_df['labels'] = train_df['labels'].map(label_map)
dev_df['labels']   = dev_df['labels'].map(label_map)

# drop invalid rows
train_df = train_df.dropna(subset=['labels'])
dev_df   = dev_df.dropna(subset=['labels'])

train_df['labels'] = train_df['labels'].astype(int)
dev_df['labels']   = dev_df['labels'].astype(int)

print("Train label distribution:")
print(train_df['labels'].value_counts())
print("Train size:", len(train_df))
print("Dev size:", len(dev_df))

# ---------- TOKENIZER ----------
MAX_VOCAB = 20000
MAX_LEN   = 50

from collections import Counter

def build_vocab(texts):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    vocab = {w: i+2 for i,(w,_) in enumerate(counter.most_common(MAX_VOCAB))}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

vocab = build_vocab(train_df['Text'])
VOCAB_SIZE = len(vocab)

def encode(text):
    tokens = str(text).split()
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

# ---------- DATASET ----------
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Text'].values
        self.labels = df['labels'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_loader = DataLoader(
    SarcasmDataset(train_df),
    batch_size=32,
    shuffle=True
)

dev_loader = DataLoader(
    SarcasmDataset(dev_df),
    batch_size=32,
    shuffle=False
)

# ---------- MODEL ----------
# ---------- MODEL ----------
class EvidentialGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)          # (B, L, E)
        _, h_n = self.gru(x)           # h_n: (1, B, H)
        h = h_n.squeeze(0)             # (B, H)

        evidence = F.softplus(self.fc(h))
        alpha = evidence + 1           # Dirichlet parameters
        return alpha



# ---------- EVIDENTIAL LOSS ----------
def evidential_loss(alpha, target):
    S = torch.sum(alpha, dim=1, keepdim=True)
    one_hot = F.one_hot(target, num_classes=2).float()
    loss = torch.sum(
        one_hot * (torch.digamma(S) - torch.digamma(alpha)),
        dim=1
    )
    return loss.mean()

# ---------- ACCURACY ----------
def compute_accuracy(alpha, targets):
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    preds = probs.argmax(dim=1)
    return (preds == targets).float().mean().item()

# ---------- EARLY STOPPING ----------
class EarlyStopping:
    def __init__(self, patience=3):
        self.best = float("inf")
        self.counter = 0
        self.patience = patience
        self.stop = False

    def step(self, val_loss):
        if val_loss < self.best:
            self.best = val_loss
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.stop = True
        return False

# ---------- TRAIN ONE MODEL ----------
def train_single_gru(seed):
    print(f"\nTraining GRU model with seed {seed}")
    set_seed(seed)

    model = EvidentialGRU(VOCAB_SIZE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    stopper = EarlyStopping(patience=3)

    save_path = os.path.join(
        RESULT_DIR, f"gru_evidential_seed_{seed}.pt"
    )

    for epoch in range(30):
        model.train()
        train_loss, train_acc = 0, 0

        for x, y in train_loader:
            optimizer.zero_grad()
            alpha = model(x)
            loss = evidential_loss(alpha, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc  += compute_accuracy(alpha, y)

        train_loss /= len(train_loader)
        train_acc  /= len(train_loader)

        model.eval()
        val_loss, val_acc = 0, 0

        with torch.no_grad():
            for x, y in dev_loader:
                alpha = model(x)
                val_loss += evidential_loss(alpha, y).item()
                val_acc  += compute_accuracy(alpha, y)

        val_loss /= len(dev_loader)
        val_acc  /= len(dev_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss {train_loss:.4f}, Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f}, Acc {val_acc:.4f}"
        )

        if stopper.step(val_loss):
            torch.save(model.state_dict(), save_path)
            print("   ✓ Best model saved")

        if stopper.stop:
            print("   ⛔ Early stopping")
            break

    return save_path



# ---------- DEEP ENSEMBLE ----------

SEEDS = [1, 7, 21, 42, 100]

gru_paths = []
for seed in SEEDS:
    path = train_single_gru(seed)
    gru_paths.append(path)

print("\nAll GRU ensemble models saved:")
for p in gru_paths:
    print(p)


Train label distribution:
labels
0    9798
1    2259
Name: count, dtype: int64
Train size: 12057
Dev size: 3015

Training GRU model with seed 1
Epoch 01 | Train Loss 0.5371, Acc 0.8125 | Val Loss 0.5119, Acc 0.8054
   ✓ Best model saved
Epoch 02 | Train Loss 0.4547, Acc 0.8225 | Val Loss 0.4357, Acc 0.8350
   ✓ Best model saved
Epoch 03 | Train Loss 0.3588, Acc 0.8643 | Val Loss 0.4304, Acc 0.8363
   ✓ Best model saved
Epoch 04 | Train Loss 0.2777, Acc 0.8972 | Val Loss 0.4465, Acc 0.8288
Epoch 05 | Train Loss 0.2033, Acc 0.9318 | Val Loss 0.4720, Acc 0.8222
Epoch 06 | Train Loss 0.1486, Acc 0.9570 | Val Loss 0.5470, Acc 0.8225
   ⛔ Early stopping

Training GRU model with seed 7
Epoch 01 | Train Loss 0.5364, Acc 0.8083 | Val Loss 0.5113, Acc 0.8051
   ✓ Best model saved
Epoch 02 | Train Loss 0.4659, Acc 0.8159 | Val Loss 0.4572, Acc 0.8054
   ✓ Best model saved
Epoch 03 | Train Loss 0.3716, Acc 0.8536 | Val Loss 0.4408, Acc 0.8330
   ✓ Best model saved
Epoch 04 | Train Loss 0.2867, Acc

In [2]:
ensemble_models = []

for seed in SEEDS:
    model = EvidentialGRU(VOCAB_SIZE).to(DEVICE)
    path = os.path.join(RESULT_DIR, f"gru_evidential_seed_{seed}.pt")
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    ensemble_models.append(model)

print(f"Loaded {len(ensemble_models)} GRU ensemble models")


Loaded 5 GRU ensemble models


In [8]:
ensemble_models = []

for seed in [1, 7, 21, 42, 100]:
    model = EvidentialGRU(VOCAB_SIZE).to(DEVICE)

    path = os.path.join(
        RESULT_DIR, f"gru_evidential_seed_{seed}.pt"
    )

    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()

    ensemble_models.append(model)

print("Loaded ensemble models:", len(ensemble_models))


Loaded ensemble models: 5


In [9]:
# ---------- UNCERTAINTY INFERENCE ----------
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

all_preds = []
all_labels = []

all_confidence = []
all_aleatoric = []
all_epistemic = []

with torch.no_grad():
    for x, y in dev_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        probs_list = []
        aleatoric_list = []

        # Forward pass through ensemble
        for model in ensemble_models:
            alpha = model(x)                    # (B, C)
            S = alpha.sum(dim=1, keepdim=True)  # Dirichlet strength

            probs = alpha / S                   # predictive probability
            aleatoric = 2.0 / S.squeeze(1)      # aleatoric uncertainty

            probs_list.append(probs)
            aleatoric_list.append(aleatoric)

        # Stack ensemble outputs
        probs_stack = torch.stack(probs_list)        # (M, B, C)
        aleatoric_stack = torch.stack(aleatoric_list)

        # Ensemble statistics
        mean_probs = probs_stack.mean(dim=0)         # (B, C)
        epistemic = probs_stack.var(dim=0).mean(dim=1)  # (B,)

        preds = mean_probs.argmax(dim=1)

        # Store outputs
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_confidence.extend(mean_probs.max(dim=1)[0].cpu().numpy())
        all_aleatoric.extend(aleatoric_stack.mean(dim=0).cpu().numpy())
        all_epistemic.extend(epistemic.cpu().numpy())


In [12]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(all_labels, all_preds)
print("Ensemble Accuracy (DEV):", acc)


Ensemble Accuracy (DEV): 0.8431177446102819


In [11]:
import numpy as np

print("Mean confidence:", np.mean(all_confidence))
print("Mean aleatoric uncertainty:", np.mean(all_aleatoric))
print("Mean epistemic uncertainty:", np.mean(all_epistemic))

print("Aleatoric min/max:", np.min(all_aleatoric), np.max(all_aleatoric))
print("Epistemic min/max:", np.min(all_epistemic), np.max(all_epistemic))


Mean confidence: 0.80510086
Mean aleatoric uncertainty: 0.08618921
Mean epistemic uncertainty: 0.015181856
Aleatoric min/max: 0.07451467 0.35436925
Epistemic min/max: 0.000117419564 0.13585144


In [13]:
threshold = np.percentile(all_epistemic, 75)   # reject top 25% uncertain

accepted_idx = np.where(np.array(all_epistemic) < threshold)[0]

filtered_acc = accuracy_score(
    np.array(all_labels)[accepted_idx],
    np.array(all_preds)[accepted_idx]
)

print("Accuracy after rejecting uncertain samples:", filtered_acc)
print("Rejected samples:", len(all_labels) - len(accepted_idx))


Accuracy after rejecting uncertain samples: 0.891640866873065
Rejected samples: 754
